# DATA INGESTION

In [ ]:
# 1. Mount Google Drive & Import Library
import os
import requests
import pandas as pd
from google.colab import drive

# Menyambungkan Google Colab ke Google Drive
drive.mount('/content/drive', force_remount=True)

# Membuat folder khusus di Drive untuk menyimpan dataset agar rapi
base_path = '/content/drive/MyDrive/RDV_Project_Dataset'
os.makedirs(base_path, exist_ok=True)

print("Google Drive siap dan folder berhasil dibuat!")

# 2. Setup Parameter Data
months = ['01', '02', '03', '04', '05', '06']
year = '2025'
taxi_base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata"

# Menentukan kolom spesifik yang akan diambil sesuai proposal
# (trip_duration dan pickup_hour biasanya dihitung saat preprocessing)
selected_columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount'
]

# 3. Ingestion Data NYC Yellow Taxi (Jan - Jun 2025)
print("\nMemulai unduhan data NYC TLC Yellow Taxi...")
for month in months:
    file_name = f"yellow_tripdata_{year}-{month}.parquet"
    file_url = f"{taxi_base_url}_{year}-{month}.parquet"
    save_path = os.path.join(base_path, file_name)

    # Cek apakah file sudah ada di Drive agar tidak download ulang
    if not os.path.exists(save_path):
        print(f"Mengunduh {file_name}...")
        try:
            # Download file secara streaming untuk menghemat RAM
            response = requests.get(file_url, stream=True)
            response.raise_for_status()
            with open(save_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)

            # Opsional: Buka file, filter kolom, dan simpan ulang untuk menghemat storage
            # df_temp = pd.read_parquet(save_path, columns=selected_columns)
            # df_temp.to_parquet(save_path, index=False)

            print(f"Berhasil menyimpan {file_name}")
        except Exception as e:
            print(f"Gagal mengunduh {file_name}: {e}")
    else:
        print(f"{file_name} sudah ada di Google Drive, melewati proses unduhan.")

# 4. Ingestion Data Eksternal (Open-Meteo Historical Weather API)
print("\nMemulai penarikan data cuaca Open-Meteo...")
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather_params = {
    "latitude": 40.7128, # Koordinat Kota New York
    "longitude": -74.0060,
    "start_date": "2025-01-01",
    "end_date": "2025-06-30",
    "hourly": ["temperature_2m", "precipitation", "weathercode"],
    "timezone": "America/New_York"
}

weather_save_path = os.path.join(base_path, "nyc_weather_jan_jun_2025.csv")

if not os.path.exists(weather_save_path):
    try:
        weather_response = requests.get(weather_url, params=weather_params)
        weather_response.raise_for_status()
        weather_data = weather_response.json()

        # Ekstrak data JSON ke dalam Pandas DataFrame
        hourly_data = weather_data['hourly']
        df_weather = pd.DataFrame({
            "datetime": pd.to_datetime(hourly_data['time']),
            "temperature_2m": hourly_data['temperature_2m'],
            "precipitation": hourly_data['precipitation'],
            "weathercode": hourly_data['weathercode']
        })

        # Simpan ke CSV di Google Drive
        df_weather.to_csv(weather_save_path, index=False)
        print(f"Berhasil menyimpan data cuaca ke {weather_save_path}")
    except Exception as e:
        print(f"Gagal menarik data cuaca: {e}")
else:
    print("Data cuaca sudah ada di Google Drive.")

print("\n--- Proses Data Ingestion Selesai ---")

MessageError: Error: credential propagation was unsuccessful

# PREPROCESING DAN CLEANING

In [ ]:
import pandas as pd
import os
import numpy as np
from google.colab import drive

# 1. Koneksi ke Drive
drive.mount('/content/drive', force_remount=True)
base_path = '/content/drive/MyDrive/RDV_Project_Dataset'
output_path = os.path.join(base_path, 'intermediate_data')
os.makedirs(output_path, exist_ok=True)

def preprocess_and_clean():
    print("Memulai proses Preprocessing dan Cleaning...")

    # --- A. LOAD DATA CUACA ---
    weather_file = os.path.join(base_path, "nyc_weather_jan_jun_2025.csv")
    df_weather = pd.read_csv(weather_file)

    # Konversi waktu cuaca dan siapkan kunci untuk merging (tanggal + jam)
    df_weather['datetime'] = pd.to_datetime(df_weather['datetime'])
    df_weather['date'] = df_weather['datetime'].dt.date
    df_weather['hour'] = df_weather['datetime'].dt.hour

    # Memilih kolom cuaca yang diperlukan saja
    df_weather = df_weather[['date', 'hour', 'temperature_2m', 'precipitation', 'weathercode']]
    print("Data cuaca siap.")

    # --- B. PROSES DATA TAKSI PER BULAN ---
    months = ['01', '02', '03', '04', '05', '06']
    all_cleaned_data = []

    for month in months:
        file_name = f"yellow_tripdata_2025-{month}.parquet"
        file_path = os.path.join(base_path, file_name)

        if not os.path.exists(file_path):
            print(f"File {file_name} tidak ditemukan, melewati...")
            continue

        print(f"Memproses: {file_name}...")
        df = pd.read_parquet(file_path)

        # 1. Konversi Tipe Data Datetime
        df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
        df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

        # 2. Feature Engineering: Trip Duration (Menit) & Pickup Hour
        # Rumus: (Dropoff - Pickup) dalam satuan menit
        df['trip_duration'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
        df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
        df['pickup_date'] = df['tpep_pickup_datetime'].dt.date

        # 3. Data Cleaning (Filtering Anomali)
        initial_count = len(df)

        # Filter: Tarif harus positif, Jarak > 0, Durasi masuk akal (1-180 menit)
        df = df[
            (df['fare_amount'] > 0) &
            (df['trip_distance'] > 0) &
            (df['trip_duration'] > 0) & (df['trip_duration'] <= 180) &
            (df['PULocationID'].between(1, 263)) & # Validasi ID Zona NYC
            (df['DOLocationID'].between(1, 263))
        ]

        # Hapus missing values pada kolom kritikal
        df = df.dropna(subset=['tpep_pickup_datetime', 'PULocationID', 'fare_amount'])

        cleaned_count = len(df)
        print(f"Pembersihan Selesai. Data dibuang: {initial_count - cleaned_count} baris.")

        # --- C. INTEGRASI DATA CUACA ---
        # Gabungkan berdasarkan tanggal dan jam pickup
        df_merged = pd.merge(
            df,
            df_weather,
            left_on=['pickup_date', 'pickup_hour'],
            right_on=['date', 'hour'],
            how='left'
        )

        # Hapus kolom pembantu merge agar tidak duplikat
        df_merged = df_merged.drop(columns=['date', 'hour', 'pickup_date'])

        all_cleaned_data.append(df_merged)

    # --- D. PENYIMPANAN AKHIR ---
    if all_cleaned_data:
        final_df = pd.concat(all_cleaned_data, ignore_index=True)

        # Simpan ke format Parquet untuk efisiensi penyimpanan (Intermediate Data)
        final_save_path = os.path.join(output_path, 'cleaned_nyc_taxi_weather_2025.parquet')
        final_df.to_parquet(final_save_path, index=False)

        print("\n" + "="*30)
        print("PROSES CLEANING SELESAI!")
        print(f"Total baris data bersih: {len(final_df)}")
        print(f"File tersimpan di: {final_save_path}")
        print("="*30)

# Jalankan fungsi
preprocess_and_clean()

Penjelasan Komponen KodePembersihan Anomali:

Sesuai instruksi, kode ini membuang fare_amount yang bernilai negatif dan baris dengan trip_distance nol. disini juga menambahkan batasan durasi (maksimal 180 menit) agar data kemacetan ekstrem atau kesalahan sistem pencatatan tidak merusak distribusi data kalian.

Feature Engineering: Kolom trip_duration dihitung dengan selisih waktu naik dan turun dalam satuan menit, serta pickup_hour diekstrak untuk menjawab tujuan penelitian kalian mengenai pola waktu.

Integrasi Cuaca: Kita menggunakan teknik merging antara data taksi dan cuaca menggunakan kolom date dan hour sebagai kunci penghubung.

Intermediate Data: Hasil akhir disimpan kembali ke Drive dalam format Parquet. Format ini dipilih karena jauh lebih cepat dan ringan dibanding CSV untuk data berukuran besar, sesuai dengan standar Data Lake profesional.

# **total keseluruhan data bersih (21.854.072 baris).**

# JUMLAH DATASET SETIAP BULAN YELLOW TAXI

In [ ]:
import pandas as pd
import os

base_path = '/content/drive/MyDrive/RDV_Project_Dataset'
months = ['01', '02', '03', '04', '05', '06']

print("=== Jumlah Data Mentah NYC Yellow Taxi 2025 ===")
total_mentah = 0

for month in months:
    file_path = os.path.join(base_path, f"yellow_tripdata_2025-{month}.parquet")
    # Trik: Kita load 1 kolom saja agar prosesnya super cepat dan tidak makan RAM
    df_raw = pd.read_parquet(file_path, columns=['tpep_pickup_datetime'])
    jumlah_baris = len(df_raw)
    total_mentah += jumlah_baris
    print(f"Bulan {month}: {jumlah_baris} baris")

print("-" * 45)
print(f"Total Keseluruhan Data Mentah: {total_mentah} baris")

# MENAMPILKAN 10 BARIS PERTAMA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython.display import display

# Path data
base_path = '/content/drive/MyDrive/RDV_Project_Dataset'
raw_path_jan = os.path.join(base_path, 'yellow_tripdata_2025-01.parquet')
cleaned_path = os.path.join(base_path, 'intermediate_data', 'cleaned_nyc_taxi_weather_2025.parquet')

# --- 1. MELIHAT 10 BARIS DATA MENTAH (SAMPEL JANUARI) ---
print("=== 10 BARIS PERTAMA DATA MENTAH (JANUARI 2025) ===")
# Kita load sedikit saja agar cepat
df_raw_sample = pd.read_parquet(raw_path_jan).head(10)
display(df_raw_sample)

# --- 2. MELIHAT 10 BARIS DATA BERSIH (INTERMEDIATE DATA) ---
print("\n=== 10 BARIS PERTAMA DATA BERSIH (SIAP ANALISIS) ===")
# Menampilkan data yang sudah digabung dengan cuaca dan ditambah kolom durasi
df_clean_sample = pd.read_parquet(cleaned_path).head(10)
display(df_clean_sample)

# --- 3. VISUALISASI EFEK CLEANING DATA ---
print("\n=== VISUALISASI HASIL PREPROCESSING ===")
# Angka ini dari perhitungan total kita sebelumnya
total_mentah = 24083384
total_bersih = 21854072
data_dibuang = total_mentah - total_bersih

labels = ['Data Mentah\n(Sebelum Cleaning)', 'Data Bersih\n(Intermediate Data)']
values = [total_mentah, total_bersih]

# Membuat Bar Chart
plt.figure(figsize=(8, 5))
sns.barplot(x=labels, y=values, palette=['#ff9999', '#66b3ff'])
plt.title('Perbandingan Jumlah Data: Mentah vs Bersih', fontsize=14, pad=15)
plt.ylabel('Jumlah Baris Data (Puluhan Juta)', fontsize=12)

# Mematikan format scientific (e+07) agar angka jelas
plt.ticklabel_format(style='plain', axis='y')

# Menambahkan teks angka di atas bar
for i, v in enumerate(values):
    plt.text(i, v + 200000, f"{v:,}", ha='center', fontsize=11, fontweight='bold')

plt.ylim(0, 27000000) # Memperluas sumbu Y agar teks tidak terpotong
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print(f"\nKesimpulan: Dari {total_mentah:,} baris, ada {data_dibuang:,} baris anomali (tarif negatif, jarak 0, dll) yang berhasil kita buang.")

# Menghitung jumlah perjalanan taksi per bulannya dari file bersih

In [ ]:
import duckdb
import os

# Tentukan lokasi file data bersih kita
cleaned_path = '/content/drive/MyDrive/RDV_Project_Dataset/intermediate_data/cleaned_nyc_taxi_weather_2025.parquet'

print("=== BUKTI DATA JANUARI - JUNI SUDAH GABUNG ===")

# Query SQL menggunakan DuckDB langsung ke file Parquet
query = f"""
SELECT
    MONTH(tpep_pickup_datetime) AS Bulan,
    COUNT(*) AS Jumlah_Perjalanan
FROM '{cleaned_path}'
GROUP BY Bulan
ORDER BY Bulan;
"""

# Menjalankan query dan mengubah hasilnya jadi DataFrame agar rapi
hasil_cek_bulan = duckdb.query(query).df()
display(hasil_cek_bulan)

# Eksplorasi Wajib (Sanity Checks)

In [ ]:
import duckdb

cleaned_path = '/content/drive/MyDrive/RDV_Project_Dataset/intermediate_data/cleaned_nyc_taxi_weather_2025.parquet'

print("=== 1. PENGECEKAN OUTLIER EKSTREM ===")
query_outlier = f"""
SELECT
    MIN(trip_distance) as Min_Jarak, MAX(trip_distance) as Max_Jarak,
    MIN(fare_amount) as Min_Tarif, MAX(fare_amount) as Max_Tarif,
    MIN(trip_duration) as Min_Durasi, MAX(trip_duration) as Max_Durasi
FROM '{cleaned_path}';
"""
display(duckdb.query(query_outlier).df())


print("\n=== 2. VALIDASI INTEGRITAS DATA CUACA ===")
query_null_weather = f"""
SELECT
    COUNT(*) AS Total_Baris,
    SUM(CASE WHEN temperature_2m IS NULL THEN 1 ELSE 0 END) AS Cuaca_Null,
    SUM(CASE WHEN weathercode IS NULL THEN 1 ELSE 0 END) AS Weathercode_Null
FROM '{cleaned_path}';
"""
display(duckdb.query(query_null_weather).df())


print("\n=== 3. MENGINTIP ANOMALI BULAN 12 ===")
query_bulan_12 = f"""
SELECT tpep_pickup_datetime, trip_distance, fare_amount
FROM '{cleaned_path}'
WHERE MONTH(tpep_pickup_datetime) = 12
LIMIT 5;
"""
display(duckdb.query(query_bulan_12).df())

1. Pengecekan Outlier Ekstrem (Jarak & Tarif)
Filter awal sudah membuang nilai negatif dan nol. Tapi, apakah ada perjalanan yang berjarak 0.5 mil tapi tarifnya $5000? Atau durasi perjalanannya 170 menit tapi jaraknya cuma 0.1 mil? Kita perlu melihat nilai maksimumnya.

2. Validasi Merge Data Cuaca
Kita menggunakan LEFT JOIN saat menggabungkan data cuaca. Kita harus memastikan tidak ada nilai NULL pada kolom cuaca (temperature_2m, precipitation, weathercode), karena model Machine Learning sangat sensitif terhadap nilai kosong.

3. Eksekusi Anomali Waktu (Bulan 12)
Kita akan melihat sekilas data bulan 12 tersebut untuk memastikan itu memang error pencatatan.

# Tahap Analisis untuk menjawab Rumusan Masalah 1: Bagaimana distribusi jumlah perjalanan taksi berdasarkan waktu (jam dalam sehari)?

Kita akan menggunakan DuckDB untuk memfilter outlier terakhir tadi (membatasi jarak max 100 mil dan tarif max $500, serta memastikan tahun 2025) sekaligus menghitung agregasi jumlah perjalanan per jamnya.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

cleaned_path = '/content/drive/MyDrive/RDV_Project_Dataset/intermediate_data/cleaned_nyc_taxi_weather_2025.parquet'

print("=== ANALISIS 1: DISTRIBUSI PERJALANAN BERDASARKAN JAM ===")

# Query DuckDB: Filter final outlier & Agregasi per jam
query_jam = f"""
SELECT
    pickup_hour AS Jam,
    COUNT(*) AS Jumlah_Perjalanan
FROM '{cleaned_path}'
WHERE
    trip_distance <= 100
    AND fare_amount <= 500
    AND YEAR(tpep_pickup_datetime) = 2025
    AND MONTH(tpep_pickup_datetime) BETWEEN 1 AND 6
    AND temperature_2m IS NOT NULL
GROUP BY Jam
ORDER BY Jam;
"""

# Eksekusi query
df_jam = duckdb.query(query_jam).df()

# Tampilkan tabel hasilnya
display(df_jam.head(24)) # Menampilkan 24 jam penuh

# --- VISUALISASI UNTUK INSIGHT ---
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_jam, x='Jam', y='Jumlah_Perjalanan', marker='o', color='#2b5c8f', linewidth=2.5)

plt.title('Distribusi Perjalanan Taksi NYC Berdasarkan Waktu (Jan - Jun 2025)', fontsize=15, pad=15)
plt.xlabel('Jam dalam Sehari (0 - 23)', fontsize=12)
plt.ylabel('Jumlah Perjalanan', fontsize=12)
plt.xticks(range(0, 24)) # Memastikan sumbu X menampilkan semua jam 0-23
plt.ticklabel_format(style='plain', axis='y') # Matikan scientific notation
plt.grid(axis='both', linestyle='--', alpha=0.5)

# Highlight area sibuk
plt.fill_between(df_jam['Jam'], df_jam['Jumlah_Perjalanan'], color='#66b3ff', alpha=0.2)

plt.show()

1. Penjelasan Output Grafik (Distribusi Waktu)
Membaca grafik garis ini sebenarnya sangat intuitif kalau kita bayangkan ritme kehidupan warga kota besar seperti New York:

- Titik Paling Sepi (Jam Terdalam): Terjadi di sekitar pukul 04:00 - 05:00 pagi
(hanya sekitar 150 ribuan perjalanan). Wajar, karena mayoritas orang sedang tidur nyenyak.

- Fase Bangun & Berangkat (Jam 06:00 - 09:00): Grafiknya mulai menanjak tajam. Ini adalah jam sibuk pagi (morning rush) ketika orang-orang mulai berangkat ke kantor atau sekolah.

- Aktivitas Siang (Jam 10:00 - 15:00): Angkanya stabil naik perlahan di kisaran 1 juta perjalanan. Ini merepresentasikan mobilitas jam makan siang atau pertemuan bisnis.

- Puncak Kesibukan (Jam 17:00 - 18:00): Ini dia "Puncak Gunung"-nya! Di jam 5 sampai 6 sore, jumlah perjalanan mencapai titik tertinggi (sekitar 1,5 juta perjalanan). Ini adalah titik krusial jam pulang kerja (evening rush hour).

- Aktivitas Malam (Jam 19:00 - 23:00): Angkanya mulai turun, tapi masih tergolong tinggi. New York adalah kota yang tidak pernah tidur, jadi mobilitas untuk makan malam atau hiburan malam masih sangat padat.

# Bagaimana pengaruh kondisi cuaca terhadap pola perjalanan taksi? Ini juga sekaligus memenuhi kriteria penilaian "Integrasi data eksternal (+10 poin)"

membagi data berdasarkan kolom precipitation (curah hujan) dari Open-Meteo untuk melihat apakah saat hujan turun, durasi perjalanan menjadi lebih lama karena macet, atau jumlah pesanan taksi justru meningkat.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

cleaned_path = '/content/drive/MyDrive/RDV_Project_Dataset/intermediate_data/cleaned_nyc_taxi_weather_2025.parquet'

print("=== ANALISIS 2: PENGARUH CUACA TERHADAP PERJALANAN ===")

# Query DuckDB: Mengelompokkan saat Hujan vs Tidak Hujan dan menghitung rata-rata
query_cuaca = f"""
SELECT
    CASE
        WHEN precipitation > 0 THEN 'Hujan'
        ELSE 'Cerah / Tidak Hujan'
    END AS Kondisi_Cuaca,
    COUNT(*) AS Jumlah_Perjalanan,
    AVG(trip_duration) AS Rata_rata_Durasi_Menit,
    AVG(trip_distance) AS Rata_rata_Jarak_Mil
FROM '{cleaned_path}'
WHERE
    trip_distance <= 100
    AND fare_amount <= 500
    AND YEAR(tpep_pickup_datetime) = 2025
    AND MONTH(tpep_pickup_datetime) BETWEEN 1 AND 6
    AND precipitation IS NOT NULL
GROUP BY Kondisi_Cuaca;
"""

# Eksekusi query
df_cuaca = duckdb.query(query_cuaca).df()

# Tampilkan tabel hasilnya
display(df_cuaca)

# --- VISUALISASI KOMBINASI (BAR & LINE CHART) ---
fig, ax1 = plt.subplots(figsize=(10, 6))

# Bar chart untuk Jumlah Perjalanan (Sumbu Y kiri)
sns.barplot(data=df_cuaca, x='Kondisi_Cuaca', y='Jumlah_Perjalanan', color='#66b3ff', ax=ax1, width=0.5)
ax1.set_ylabel('Total Perjalanan (Puluhan Juta)', fontsize=12, color='#2b5c8f')
ax1.set_xlabel('Kondisi Cuaca', fontsize=12)
ax1.tick_params(axis='y', labelcolor='#2b5c8f')
ax1.ticklabel_format(style='plain', axis='y') # Matikan scientific notation

# Buat sumbu Y kedua (kanan) untuk Rata-rata Durasi
ax2 = ax1.twinx()
# Line chart overlay untuk Rata-rata Durasi (Sumbu Y kanan)
sns.lineplot(data=df_cuaca, x='Kondisi_Cuaca', y='Rata_rata_Durasi_Menit', color='#ff6666', marker='o', linewidth=3, markersize=10, ax=ax2)
ax2.set_ylabel('Rata-rata Durasi (Menit)', fontsize=12, color='#cc0000')
ax2.tick_params(axis='y', labelcolor='#cc0000')

# Set batas bawah sumbu Y kanan agar proporsional
ax2.set_ylim(bottom=df_cuaca['Rata_rata_Durasi_Menit'].min() - 2)

plt.title('Pengaruh Kondisi Cuaca terhadap Volume dan Durasi Taksi (Jan - Jun 2025)', fontsize=14, pad=15)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.show()

1. Volume Perjalanan (Grafik Batang Biru)
Data: Terdapat ketimpangan jumlah perjalanan yang sangat besar antara cuaca cerah (17.913.225 perjalanan) dan cuaca hujan (3.939.790 perjalanan).

- Analisis: Ini sangat logis karena secara statistik dalam periode 6 bulan (Januari - Juni), jumlah hari atau jam dengan kondisi cuaca cerah di New York jauh lebih banyak dibandingkan waktu turunnya hujan. Grafik batang biru ini merepresentasikan total populasi data kita berdasarkan kondisi cuaca.

2. Rata-rata Durasi Perjalanan (Grafik Garis Merah)
Data: Rata-rata durasi perjalanan saat cuaca cerah adalah 16,01 menit, sedangkan saat hujan naik menjadi 16,80 menit.

- Analisis: Garis merah yang menanjak naik dari kiri ke kanan menunjukkan fakta penting: Hujan memperlambat laju lalu lintas. Meskipun selisihnya hanya sekitar 0,8 menit (sekitar 48 detik) secara rata-rata global, ini membuktikan hipotesis bahwa kondisi jalanan yang basah, jarak pandang yang menurun, atau kemacetan tambahan akibat hujan membuat perjalanan taksi memakan waktu lebih lama.

3. Rata-rata Jarak Tempuh (Dari Tabel)
Data: Rata-rata jarak tempuh justru sedikit menurun saat hujan (3,22 mil) dibandingkan saat cerah (3,33 mil).

- Analisis: Ini adalah temuan perilaku penumpang yang sangat menarik! Saat hujan turun, orang-orang cenderung memesan taksi untuk jarak dekat yang biasanya mungkin mereka tempuh dengan berjalan kaki santai atau menggunakan sepeda/skuter sewaan. Mereka tidak ingin kehujanan, sehingga rela membayar taksi untuk rute pendek.

Kesimpulan Akhir
Kondisi hujan di Kota New York memicu anomali perilaku mobilitas. Penumpang cenderung menggunakan taksi untuk rute yang lebih pendek demi menghindari hujan, namun perjalanan tersebut memakan waktu yang lebih lama akibat penurunan kecepatan lalu lintas secara keseluruhan.

# sama aja sii cuman mau lihat grafik yang lebih gampang

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

cleaned_path = '/content/drive/MyDrive/RDV_Project_Dataset/intermediate_data/cleaned_nyc_taxi_weather_2025.parquet'

print("=== ANALISIS 2: PENGARUH CUACA TERHADAP PERJALANAN (VERSI MUDAH DIPAHAMI) ===")

# Query DuckDB (Sama seperti sebelumnya)
query_cuaca = f"""
SELECT
    CASE
        WHEN precipitation > 0 THEN 'Hujan'
        ELSE 'Cerah / Tidak Hujan'
    END AS Kondisi_Cuaca,
    COUNT(*) AS Jumlah_Perjalanan,
    AVG(trip_duration) AS Rata_rata_Durasi_Menit,
    AVG(trip_distance) AS Rata_rata_Jarak_Mil
FROM '{cleaned_path}'
WHERE
    trip_distance <= 100
    AND fare_amount <= 500
    AND YEAR(tpep_pickup_datetime) = 2025
    AND MONTH(tpep_pickup_datetime) BETWEEN 1 AND 6
    AND precipitation IS NOT NULL
GROUP BY Kondisi_Cuaca;
"""

df_cuaca = duckdb.query(query_cuaca).df()

# --- VISUALISASI YANG LEBIH INTUITIF (2 GRAFIK BERDAMPINGAN) ---
# Menggunakan warna yang menggambarkan cuaca
warna_cuaca = {'Cerah / Tidak Hujan': '#FFB347', 'Hujan': '#5DADE2'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Dampak Kondisi Cuaca pada Taksi NYC (Jan - Jun 2025)', fontsize=16, fontweight='bold', y=1.05)

# --- GRAFIK 1: JUMLAH PERJALANAN ---
sns.barplot(data=df_cuaca, x='Kondisi_Cuaca', y='Jumlah_Perjalanan', palette=warna_cuaca, ax=ax1)
ax1.set_title('1. Perbandingan Volume Pesanan', fontsize=14, pad=10)
ax1.set_xlabel('')
ax1.set_ylabel('Total Perjalanan', fontsize=12)
ax1.ticklabel_format(style='plain', axis='y')
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# Menambahkan teks angka di atas bar Grafik 1
for p in ax1.patches:
    ax1.annotate(f"{int(p.get_height()):,}",
                 (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=12, fontweight='bold', color='black', xytext=(0, 5), textcoords='offset points')

# --- GRAFIK 2: RATA-RATA DURASI ---
sns.barplot(data=df_cuaca, x='Kondisi_Cuaca', y='Rata_rata_Durasi_Menit', palette=warna_cuaca, ax=ax2)
ax2.set_title('2. Perbandingan Lama Perjalanan (Menit)', fontsize=14, pad=10)
ax2.set_xlabel('')
ax2.set_ylabel('Rata-rata Durasi (Menit)', fontsize=12)
ax2.set_ylim(0, 19) # Membesarkan batas atas agar label teks tidak terpotong
ax2.grid(axis='y', linestyle='--', alpha=0.5)

# Menambahkan teks angka di atas bar Grafik 2
for p in ax2.patches:
    ax2.annotate(f"{p.get_height():.2f} Menit",
                 (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=12, fontweight='bold', color='red', xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.show()

# Analisis Geospasial (Spasial)

menjawab rumusan masalah: "Zona mana yang memiliki aktivitas penjemputan tertinggi?".Masalahnya, di dataset, lokasi penjemputan hanya berupa angka (PULocationID), misalnya ID 237. Angka ini tidak akan ada artinya di peta kalau kita tidak tahu itu nama daerah apa. Oleh karena itu, harus menarik satu file tambahan berukuran kecil dari NYC TLC, yaitu taxi_zone_lookup.csv, untuk menerjemahkan ID tersebut menjadi nama daerah yang sesungguhnya (misal: Upper East Side atau JFK Airport).

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

cleaned_path = '/content/drive/MyDrive/RDV_Project_Dataset/intermediate_data/cleaned_nyc_taxi_weather_2025.parquet'

print("=== ANALISIS 3: 10 ZONA PENJEMPUTAN TAKSI TERTINGGI (PERSIAPAN PETA) ===")

# 1. Ambil data kamus zona dari server NYC TLC
url_zone = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
df_zone = pd.read_csv(url_zone)

# 2. Query menghitung jumlah penjemputan per ID Lokasi
query_zona = f"""
SELECT
    PULocationID AS ID_Lokasi,
    COUNT(*) AS Jumlah_Penjemputan
FROM '{cleaned_path}'
WHERE
    trip_distance <= 100
    AND fare_amount <= 500
    AND YEAR(tpep_pickup_datetime) = 2025
    AND MONTH(tpep_pickup_datetime) BETWEEN 1 AND 6
GROUP BY ID_Lokasi
ORDER BY Jumlah_Penjemputan DESC
LIMIT 10;
"""

df_top_zona = duckdb.query(query_zona).df()

# 3. Gabungkan hasil hitungan dengan nama zonanya
df_final_zona = pd.merge(df_top_zona, df_zone, left_on='ID_Lokasi', right_on='LocationID', how='left')

# Rapikan kolom yang ditampilkan
df_final_zona = df_final_zona[['Borough', 'Zone', 'Jumlah_Penjemputan']]
display(df_final_zona)

# --- VISUALISASI HORIZONTAL BAR CHART ---
plt.figure(figsize=(10, 6))
sns.barplot(data=df_final_zona, x='Jumlah_Penjemputan', y='Zone', hue='Borough', dodge=False, palette='viridis')

plt.title('Top 10 Zona Penjemputan Taksi Tertinggi di NYC (Jan - Jun 2025)', fontsize=14, pad=15)
plt.xlabel('Total Penjemputan', fontsize=12)
plt.ylabel('Nama Zona', fontsize=12)
plt.ticklabel_format(style='plain', axis='x')
plt.grid(axis='x', linestyle='--', alpha=0.5)

plt.show()

1. Dominasi Absolut Manhattan
Dari 10 zona tertinggi, 8 di antaranya berada di wilayah (Borough) Manhattan. Ini sangat masuk akal karena Manhattan adalah pusat ekonomi, bisnis, dan hiburan utama di New York. Zona perumahan elit seperti Upper East Side South dan North menduduki peringkat teratas (lebih dari 1 juta dan 897 ribu penjemputan), kemungkinan karena penduduk di area ini memiliki daya beli tinggi dan mengandalkan taksi untuk mobilitas harian mereka.

2. Anomali Logis di Queens (Pusat Bandara)
Wilayah Queens diwakili oleh warna hijau. Menariknya, dua area di Queens yang berhasil masuk ke top 10 ini keduanya adalah bandara, yaitu JFK Airport (peringkat 4) dan LaGuardia Airport (peringkat 9). Ini membuktikan bahwa selain pusat bisnis, titik transit udara adalah generator utama bagi tingginya permintaan taksi.

3. Titik Kumpul Turis dan Pekerja
Selain area perumahan elit, zona Manhattan lainnya yang masuk daftar adalah Midtown Center, Times Sq/Theatre District, dan Penn Station. Ini adalah pusat pariwisata dunia dan stasiun transit kereta api tersibuk. Jadi, mobilitas penumpang taksi di sini sangat dipengaruhi oleh wisatawan dan komuter yang baru turun dari kereta.